In [ ]:
# CMP7005 Programming for Data Analysis
## Task 3 – Machine Learning Model Development

In [1]:
import pandas as pd
import numpy as np

cleaned_df = pd.read_csv(
    "../data/Loan_Approval_Cleaned.csv",
    parse_dates=["ApplicationDate"]
)

print("Dataset shape:", cleaned_df.shape)
cleaned_df.head()

Dataset shape: (20000, 34)


,ID,ApplicationDate,AnnualIncome,EmploymentStatus,LoanDuration,NumberOfDependents,MonthlyDebtPayments,NumberOfOpenCreditLines,BankruptcyHistory,PreviousLoanDefaults,...,PaymentHistory,SavingsAccountBalance,TotalAssets,JobTenure,InterestRate,RiskScore,Experience,MonthlyIncome,MonthlyLoanPayment,TotalDebtToIncomeRatio
0,A001,2018-01-01,39948,Employed,48,2,183,1,0,0,...,29,7632,146111,11,22.75,49.0,22.0,3329.00,419.74,18.11
1,A002,2018-01-02,39709,Employed,48,1,496,5,0,0,...,21,4627,53204,3,20.10,51.0,15.0,3309.08,793.95,38.98
2,A003,2018-01-03,40724,Employed,36,2,902,2,0,0,...,20,886,25176,6,21.25,51.0,26.0,3393.67,666.36,46.21
3,A004,2018-01-04,69084,Employed,96,1,755,2,0,0,...,27,1675,104822,5,30.09,54.0,34.0,5757.00,1047.48,31.31
4,A005,2018-01-05,103264,Employed,36,1,274,0,0,0,...,26,1555,244305,5,17.59,36.0,17.0,8605.33,330.14,7.02


In [2]:
print(cleaned_df.columns.tolist())

['ID', 'ApplicationDate', 'AnnualIncome', 'EmploymentStatus', 'LoanDuration', 'NumberOfDependents', 'MonthlyDebtPayments', 'NumberOfOpenCreditLines', 'BankruptcyHistory', 'PreviousLoanDefaults', 'LengthOfCreditHistory', 'CheckingAccountBalance', 'TotalLiabilities', 'NetWorth', 'LoanApproved', 'Age', 'CreditScore', 'EducationLevel', 'LoanAmount', 'MaritalStatus', 'HomeOwnershipStatus', 'CreditCardUtilizationRate', 'NumberOfCreditInquiries', 'LoanPurpose', 'PaymentHistory', 'SavingsAccountBalance', 'TotalAssets', 'JobTenure', 'InterestRate', 'RiskScore', 'Experience', 'MonthlyIncome', 'MonthlyLoanPayment', 'TotalDebtToIncomeRatio']


In [3]:
for i, col in enumerate(cleaned_df.columns, start=1):
    print(i, col)

1 ID
2 ApplicationDate
3 AnnualIncome
4 EmploymentStatus
5 LoanDuration
6 NumberOfDependents
7 MonthlyDebtPayments
8 NumberOfOpenCreditLines
9 BankruptcyHistory
10 PreviousLoanDefaults
11 LengthOfCreditHistory
12 CheckingAccountBalance
13 TotalLiabilities
14 NetWorth
15 LoanApproved
16 Age
17 CreditScore
18 EducationLevel
19 LoanAmount
20 MaritalStatus
21 HomeOwnershipStatus
22 CreditCardUtilizationRate
23 NumberOfCreditInquiries
24 LoanPurpose
25 PaymentHistory
26 SavingsAccountBalance
27 TotalAssets
28 JobTenure
29 InterestRate
30 RiskScore
31 Experience
32 MonthlyIncome
33 MonthlyLoanPayment
34 TotalDebtToIncomeRatio


In [ ]:
### 3.1 Feature Selection

In [4]:
drop_columns = [
    "ID",
    "ApplicationDate",
    "MonthlyIncome",
    "LoanApproved"
]

X = cleaned_df.drop(columns=drop_columns)
y = cleaned_df["LoanApproved"]

In [5]:
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

X shape: (20000, 30)
y shape: (20000,)

Features:
['AnnualIncome', 'EmploymentStatus', 'LoanDuration', 'NumberOfDependents', 'MonthlyDebtPayments', 'NumberOfOpenCreditLines', 'BankruptcyHistory', 'PreviousLoanDefaults', 'LengthOfCreditHistory', 'CheckingAccountBalance', 'TotalLiabilities', 'NetWorth', 'Age', 'CreditScore', 'EducationLevel', 'LoanAmount', 'MaritalStatus', 'HomeOwnershipStatus', 'CreditCardUtilizationRate', 'NumberOfCreditInquiries', 'LoanPurpose', 'PaymentHistory', 'SavingsAccountBalance', 'TotalAssets', 'JobTenure', 'InterestRate', 'RiskScore', 'Experience', 'MonthlyLoanPayment', 'TotalDebtToIncomeRatio']


In [6]:
print(y.value_counts())

print("\nPercentage:")
print(
    y.value_counts(normalize=True).mul(100).round(2)
)

LoanApproved
0    15220
1     4780
Name: count, dtype: int64

Percentage:
LoanApproved
0    76.1
1    23.9
Name: proportion, dtype: float64


In [13]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Numerical features:", len(numerical_features))
print(numerical_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

Numerical features: 25
['AnnualIncome', 'LoanDuration', 'NumberOfDependents', 'MonthlyDebtPayments', 'NumberOfOpenCreditLines', 'BankruptcyHistory', 'PreviousLoanDefaults', 'LengthOfCreditHistory', 'CheckingAccountBalance', 'TotalLiabilities', 'NetWorth', 'Age', 'CreditScore', 'LoanAmount', 'CreditCardUtilizationRate', 'NumberOfCreditInquiries', 'PaymentHistory', 'SavingsAccountBalance', 'TotalAssets', 'JobTenure', 'InterestRate', 'RiskScore', 'Experience', 'MonthlyLoanPayment', 'TotalDebtToIncomeRatio']

Categorical features: 5
['EmploymentStatus', 'EducationLevel', 'MaritalStatus', 'HomeOwnershipStatus', 'LoanPurpose']


In [14]:
from sklearn.model_selection import train_test_split

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [16]:
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTesting target distribution:")
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Training features: (16000, 30)
Testing features: (4000, 30)

Training target distribution:
LoanApproved
0    76.1
1    23.9
Name: proportion, dtype: float64

Testing target distribution:
LoanApproved
0    76.1
1    23.9
Name: proportion, dtype: float64


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

In [ ]:
### 3.2 Train-Test Split

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [20]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (16000, 30)
X_test shape: (4000, 30)
y_train shape: (16000,)
y_test shape: (4000,)


In [22]:
print("Training target distribution:")
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTesting target distribution:")
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Training target distribution:
LoanApproved
0    76.1
1    23.9
Name: proportion, dtype: float64

Testing target distribution:
LoanApproved
0    76.1
1    23.9
Name: proportion, dtype: float64


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

In [ ]:
### 3.3 Feature Preprocessing

### Numerical variables are standardised using `StandardScaler`, while categorical variables are transformed using one-hot encoding. One-hot encoding avoids introducing an artificial ordinal relationship between nominal categories. Unknown categories are ignored during transformation so that previously unseen categories do not cause prediction errors.

In [ ]:
### 3.4 Decision Tree Classifier

### A Decision Tree Classifier is developed as the first supervised classification algorithm. Decision trees recursively divide observations according to predictor values and produce interpretable decision rules for classifying loan applications as approved or rejected.

### A baseline model is initially trained before hyperparameter tuning is performed.

In [23]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

In [24]:
decision_tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(
                random_state=42
            )
        )
    ]
)

In [25]:
decision_tree_pipeline.fit(
    X_train,
    y_train
)

print("Decision Tree training completed.")

Decision Tree training completed.


In [26]:
y_train_pred_dt = decision_tree_pipeline.predict(
    X_train
)

y_test_pred_dt = decision_tree_pipeline.predict(
    X_test
)

In [27]:
from sklearn.metrics import accuracy_score

In [28]:
train_accuracy_dt = accuracy_score(
    y_train,
    y_train_pred_dt
)

test_accuracy_dt = accuracy_score(
    y_test,
    y_test_pred_dt
)

print(
    "Decision Tree Training Accuracy:",
    round(train_accuracy_dt, 4)
)

print(
    "Decision Tree Testing Accuracy:",
    round(test_accuracy_dt, 4)
)

Decision Tree Training Accuracy: 1.0
Decision Tree Testing Accuracy: 0.9792


In [29]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [30]:
dt_accuracy = accuracy_score(
    y_test,
    y_test_pred_dt
)

dt_precision = precision_score(
    y_test,
    y_test_pred_dt
)

dt_recall = recall_score(
    y_test,
    y_test_pred_dt
)

dt_f1 = f1_score(
    y_test,
    y_test_pred_dt
)

print("Decision Tree Test Metrics")
print("--------------------------")
print("Accuracy :", round(dt_accuracy, 4))
print("Precision:", round(dt_precision, 4))
print("Recall   :", round(dt_recall, 4))
print("F1-score :", round(dt_f1, 4))

Decision Tree Test Metrics
--------------------------
Accuracy : 0.9792
Precision: 0.9552
Recall   : 0.9582
F1-score : 0.9567


In [31]:
print(
    classification_report(
        y_test,
        y_test_pred_dt,
        target_names=["Rejected", "Approved"]
    )
)

              precision    recall  f1-score   support

    Rejected       0.99      0.99      0.99      3044
    Approved       0.96      0.96      0.96       956

    accuracy                           0.98      4000
   macro avg       0.97      0.97      0.97      4000
weighted avg       0.98      0.98      0.98      4000



In [32]:
cm_dt = confusion_matrix(
    y_test,
    y_test_pred_dt
)

print(cm_dt)

[[3001   43]
 [  40  916]]


In [33]:
print("Training target distribution:")
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTesting target distribution:")
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Training target distribution:
LoanApproved
0    76.1
1    23.9
Name: proportion, dtype: float64

Testing target distribution:
LoanApproved
0    76.1
1    23.9
Name: proportion, dtype: float64


In [34]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [ ]:
### 3.4 Decision Tree Classifier

### A Decision Tree Classifier is used as the first supervised machine-learning classification algorithm. A baseline model is initially trained and evaluated before hyperparameter tuning is performed.

In [35]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

decision_tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(random_state=42)
        )
    ]
)

In [36]:
decision_tree_pipeline.fit(X_train, y_train)

print("Decision Tree training completed.")

Decision Tree training completed.


In [37]:
y_train_pred_dt = decision_tree_pipeline.predict(X_train)
y_test_pred_dt = decision_tree_pipeline.predict(X_test)

In [38]:
from sklearn.metrics import accuracy_score

train_accuracy_dt = accuracy_score(
    y_train,
    y_train_pred_dt
)

test_accuracy_dt = accuracy_score(
    y_test,
    y_test_pred_dt
)

print(
    "Decision Tree Training Accuracy:",
    round(train_accuracy_dt, 4)
)

print(
    "Decision Tree Testing Accuracy:",
    round(test_accuracy_dt, 4)
)

Decision Tree Training Accuracy: 1.0
Decision Tree Testing Accuracy: 0.9792


In [39]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

dt_accuracy = accuracy_score(
    y_test,
    y_test_pred_dt
)

dt_precision = precision_score(
    y_test,
    y_test_pred_dt
)

dt_recall = recall_score(
    y_test,
    y_test_pred_dt
)

dt_f1 = f1_score(
    y_test,
    y_test_pred_dt
)

print("Decision Tree Test Metrics")
print("--------------------------")
print("Accuracy :", round(dt_accuracy, 4))
print("Precision:", round(dt_precision, 4))
print("Recall   :", round(dt_recall, 4))
print("F1-score :", round(dt_f1, 4))

Decision Tree Test Metrics
--------------------------
Accuracy : 0.9792
Precision: 0.9552
Recall   : 0.9582
F1-score : 0.9567


In [40]:
print(
    classification_report(
        y_test,
        y_test_pred_dt,
        target_names=["Rejected", "Approved"]
    )
)

              precision    recall  f1-score   support

    Rejected       0.99      0.99      0.99      3044
    Approved       0.96      0.96      0.96       956

    accuracy                           0.98      4000
   macro avg       0.97      0.97      0.97      4000
weighted avg       0.98      0.98      0.98      4000



In [41]:
cm_dt = confusion_matrix(
    y_test,
    y_test_pred_dt
)

print("Confusion Matrix:")
print(cm_dt)

Confusion Matrix:
[[3001   43]
 [  40  916]]


In [ ]:
#### Baseline Decision Tree Interpretation

### The baseline Decision Tree achieved a training accuracy of 100% and a testing accuracy of approximately 97.92%. Although the test performance is high, the perfect training accuracy suggests that the unrestricted tree may have overfitted the training data.

### On the test set, the model achieved approximately 95.52% precision, 95.82% recall, and an F1-score of 95.67% for the approved class. The confusion matrix shows that 3,001 rejected and 916 approved applications were classified correctly, while 43 rejected applications were incorrectly classified as approved and 40 approved applications were incorrectly classified as rejected.

### The strong recall for the minority approved class is particularly important given the class imbalance identified during exploratory analysis. Hyperparameter tuning will therefore be performed to determine whether a simpler Decision Tree can maintain strong predictive performance while reducing overfitting.

In [42]:
from sklearn.model_selection import GridSearchCV

In [43]:
dt_param_grid = {
    "classifier__criterion": [
        "gini",
        "entropy"
    ],
    "classifier__max_depth": [
        5,
        10,
        15,
        20,
        None
    ],
    "classifier__min_samples_split": [
        2,
        5,
        10
    ],
    "classifier__min_samples_leaf": [
        1,
        2,
        5
    ]
}

In [44]:
dt_grid_search = GridSearchCV(
    estimator=decision_tree_pipeline,
    param_grid=dt_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

In [45]:
dt_grid_search.fit(
    X_train,
    y_train
)

Fitting 5 folds for each of 90 candidates, totalling 450 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__criterion': ['gini', 'entropy'], 'classifier__max_depth': [5, 10, ...], 'classifier__min_samples_leaf': [1, 2, ...], 'classifier__min_samples_split': [2, 5, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about mul

In [46]:
print(
    "Best Parameters:"
)

print(
    dt_grid_search.best_params_
)

print(
    "\nBest Cross-Validation F1:",
    round(dt_grid_search.best_score_, 4)
)

Best Parameters:
{'classifier__criterion': 'entropy', 'classifier__max_depth': 15, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 10}

Best Cross-Validation F1: 0.9675


In [47]:
best_dt_model = dt_grid_search.best_estimator_

In [48]:
y_test_pred_dt_tuned = best_dt_model.predict(
    X_test
)

In [49]:
dt_tuned_accuracy = accuracy_score(
    y_test,
    y_test_pred_dt_tuned
)

dt_tuned_precision = precision_score(
    y_test,
    y_test_pred_dt_tuned
)

dt_tuned_recall = recall_score(
    y_test,
    y_test_pred_dt_tuned
)

dt_tuned_f1 = f1_score(
    y_test,
    y_test_pred_dt_tuned
)

print("Tuned Decision Tree Test Metrics")
print("--------------------------------")
print(
    "Accuracy :",
    round(dt_tuned_accuracy, 4)
)
print(
    "Precision:",
    round(dt_tuned_precision, 4)
)
print(
    "Recall   :",
    round(dt_tuned_recall, 4)
)
print(
    "F1-score :",
    round(dt_tuned_f1, 4)
)

Tuned Decision Tree Test Metrics
--------------------------------
Accuracy : 0.9858
Precision: 0.9687
Recall   : 0.9718
F1-score : 0.9702


In [50]:
cm_dt_tuned = confusion_matrix(
    y_test,
    y_test_pred_dt_tuned
)

print("Confusion Matrix:")
print(cm_dt_tuned)

Confusion Matrix:
[[3014   30]
 [  27  929]]


In [ ]:
#### Decision Tree Hyperparameter Tuning Results

### Hyperparameter tuning improved the Decision Tree's predictive performance. Grid search with five-fold cross-validation selected entropy as the splitting criterion, a maximum tree depth of 15, a minimum of 10 samples required to split an internal node, and a minimum leaf size of 1.

### The tuned model achieved an accuracy of approximately 98.58%, precision of 96.87%, recall of 97.18%, and an F1-score of 97.02% on the test dataset. This represents an improvement over the baseline Decision Tree, which achieved an F1-score of approximately 95.67%.

### The tuned model also reduced false-positive predictions from 43 to 30 and false-negative predictions from 40 to 27. The cross-validation F1-score of approximately 96.75% is close to the test-set F1-score of 97.02%, providing evidence that the tuned model generalises well to unseen observations.